# Conversión de dataset a formato YOLOv8 (segmentación)
Este notebook procesa un conjunto de imágenes y máscaras para generar etiquetas de segmentación compatibles con YOLOv8/YOLOv9.

Las imágenes con prefijo `colonca` se consideran **cancerosas (clase 0)** y las con prefijo `colonn` son **benignas (clase 1)**.

In [3]:
!pip install opencv-python-headless



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 926.0 kB/s eta 0:00:000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 5.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 5.8 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-learn 1.3.1 requires numpy<2.0,>=1.17.3, but you have numpy 2.2.6 which is incompatible.
scipy 1.11.3 requires numpy<1.28.0,>=1.21.6, but you have numpy 2.2.6 which is incompatible.
matplotlib 3.8.0 requires numpy<2,>=1.21, but you have numpy 2.2.6 which is incompatible.
contourpy 1.1.1 requires numpy<2.0,>=1.16; python_version <= "3.11", but you have numpy 2.2.6 which is incompatible.
numba 0.57.1 requ

In [2]:
import cv2
import os
import numpy as np
from tqdm import tqdm
import random
import shutil

In [4]:
# === CONFIGURACIÓN BÁSICA ===
base_dir = "../dataset"
image_dir = os.path.join(base_dir, "images")
mask_dir = os.path.join(base_dir, "masks")

train_img_dir = os.path.join(base_dir, "images/train")
val_img_dir = os.path.join(base_dir, "images/val")
train_lbl_dir = os.path.join(base_dir, "labels/train")
val_lbl_dir = os.path.join(base_dir, "labels/val")

# Crear carpetas necesarias
for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(d, exist_ok=True)

In [5]:
VAL_SPLIT = 0.2  # 20% para validación
random.seed(42)

CLASS_MAP = {
    "colonca": 0,  # clase 0 = canceroso
    "colonn": 1,   # clase 1 = benigno
}

In [6]:
# === FUNCIONES AUXILIARES ===
def get_prefix(filename: str):
    name = os.path.splitext(filename)[0]
    prefix = ''.join([c for c in name if not c.isdigit()])
    return prefix.lower()

def write_yolo_segmentation(contours, class_id, w, h, label_path):
    with open(label_path, 'w') as f:
        for contour in contours:
            if len(contour) < 6:
                continue
            if cv2.contourArea(contour) < 20:
                continue
            contour = contour.squeeze(1)
            norm = []
            for x, y in contour:
                norm.append(f"{x / w:.6f}")
                norm.append(f"{y / h:.6f}")
            f.write(f"{class_id} " + " ".join(norm) + "\n")

def process_split(split_images, img_dest, lbl_dest):
    for filename in tqdm(split_images, desc=f"Procesando {os.path.basename(img_dest)}"):
        prefix = get_prefix(filename)
        if prefix not in CLASS_MAP:
            print(f"{filename}: prefijo '{prefix}' no reconocido, se omite.")
            continue

        class_id = CLASS_MAP[prefix]
        img_path = os.path.join(image_dir, filename)
        mask_path = os.path.join(mask_dir, filename)

        if not os.path.exists(mask_path):
            print(f"No se encontró la máscara para {filename}, se omite.")
            continue

        shutil.copy(img_path, os.path.join(img_dest, filename))

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            print(f"Error leyendo máscara: {mask_path}")
            continue

        h, w = mask.shape
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        label_path = os.path.join(lbl_dest, os.path.splitext(filename)[0] + ".txt")
        write_yolo_segmentation(contours, class_id, w, h, label_path)

In [7]:
# === DIVISIÓN TRAIN/VAL ===
images = [f for f in os.listdir(image_dir) if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))]
random.shuffle(images)

split_idx = int(len(images) * (1 - VAL_SPLIT))
train_images = images[:split_idx]
val_images = images[split_idx:]

In [8]:
# === PROCESAMIENTO ===
print("Generando conjunto de entrenamiento...")
process_split(train_images, train_img_dir, train_lbl_dir)

print("Generando conjunto de validación...")
process_split(val_images, val_img_dir, val_lbl_dir)

print("Conversión a formato YOLOv8 completada.")

Generando conjunto de entrenamiento...


Procesando train: 100%|██████████| 8000/8000 [12:36<00:00, 10.57it/s]  


Generando conjunto de validación...


Procesando val: 100%|██████████| 2000/2000 [03:22<00:00,  9.88it/s]

Conversión a formato YOLOv8 completada.
